# Machine learning-based surrogate modeling with application to a tsunami case

## Load modules

The tutorial is based on the python module [scikit-learn](https://scikit-learn.org/): a Python module integrating a wide range of state-of-the-art machine learning algorithms for medium-scale supervised and unsupervised problems. See details [Pedregosa et al. (2011)](http://www.jmlr.org/papers/volume12/pedregosa11a/pedregosa11a.pdf?source=post_page).

In [ ]:
# load specific functions for manipulating, plotting data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# load specific functions of sklearn
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn import model_selection
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Import all regression models you want to test
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

## Context

The application test case is adapted from the Ligurian (North West of Italy) 1887 tsunami case in particular studied by [Larroque et al. (2012)](https://academic.oup.com/gji/article/190/1/87/601785). The following figure provides the structural scheme of the fault rupture mechanism.

![ligurianEarthquake.png](./data/ligurianEarthquake.png).

We will focus on the impact of the uncertainties of the earthquake source parameters on the the sea surface elevatio denoted *SSH* induced by the tsunami. Nine uncertain parameters are considered:
* $X_{1} = Lon$, Longitude of the  epicentral position (°)
* $X_{2} = Lat$, Latitude of the  epicentral position (°)
* $X_{3} = Z$, Depth  (m)
* $X_{4} = Azi$, Azimuth  (°)
* $X_{5} = Dip$, Dip angle  (m)
* $X_{6} = Rake$, Rake angle  (°)
* $X_{7} = D$, average slip on the fault surface  (m)
* $X_{8} = W$, Width of the fault surface  (m)
* $X_{9} = L$, Length of the fault surface  (m)

A numerical model has been set up to estimate the tsunami-induced sea surface elevation at different locations along the French and Italian coast from the values of the earthquake source parameters. See details in [Rohmer et al. 2018](https://link.springer.com/article/10.1007/s00477-017-1423-y).
The model has $8$ input random variables. A set of 300 random samples was generated by assuming that all input variables vary uniformly between the lower and upper bound described in the Table below.
![Table](./data/Assumption.png)
For each input sample, a numerical model is run to compute the response (sea surface elevation) at all points along the coast.
![location.png](./data/location.png)

## Objectives
The objective is to model the mathematical link between the input parameters *X* and *SSH* using a machine learning model. 

In this tutorial, you will:
* train a [random forest regression model](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) and assess its prediction capability via cross validation procedure
* select the most optimal hyper-parameters of this model
* compare to alternative machine learning models
* select the most optimal model and tune its hyper-parameters

## Data

In [ ]:
##Load the inputs
df = pd.read_csv("./data/DOE_aziPetit.txt",sep=" ")
X = df.values
d = df.shape[1] # number of inputs
N = df.shape[0] # number of samples

**QUESTION 1** Create cloud pairs graphs to visualise the input values and their distribution using [seaborn pairplot](https://seaborn.pydata.org/generated/seaborn.pairplot.html). What can you say about the distribution of the inputs?

In [ ]:
sns.pairplot(df)

In [ ]:
##Load the response (output)
outputSample0  = pd.read_csv("./data/Hmax_aziPetit.txt",sep=" ", header=None)

We wil first focus on location #8 =  Imperia (see map above).

In [ ]:
loc = 8
Y = outputSample0.values[:,loc]
TARGET_LABEL = ["SSH" + str(loc) + ' [m]']

We plot here the histogram of output *Y*.

**QUESTION 2** what can you say about the characteristics of *Y*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(Y, bins=30, edgecolor="white", alpha=0.85, color="#6a9bcc")
ax.axvline(np.mean(Y), color="#d97757", linestyle="--", linewidth=2, label=f"Mean = {np.mean(Y):.1f}")
ax.axvline(np.median(Y), color="#141413", linestyle=":", linewidth=2, label=f"Median = {np.median(Y):.1f}")
ax.set_xlabel(TARGET_LABEL); ax.set_ylabel("Count")
ax.set_title("Target variable")
ax.legend()
#plt.savefig(IMAGES_DIR / "ml_target_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## Surrogate modelling with a machine learning model

We will first split the dataset (X,Y) in training and test subsets by following a 5-fold cross-validation procedure repeated 5 times. See details [here](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RepeatedKFold.html).
To assess the predictive performance, we choose as a scoring criterion the [r²](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html).

In [ ]:
## define the cross validation procedure
kfold = model_selection.RepeatedKFold(n_splits=2,n_repeats=1, random_state=42)
scoring = 'r2'

We will fit and test the performance of a [random forest regression model](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html).

In [ ]:
## fit the rf model
rf0 = RandomForestRegressor(random_state=0)
rf0.fit(X,Y)

In [ ]:
## perform the CV procedure
cv_results = model_selection.cross_val_score(rf0, X, Y, cv=kfold, scoring=scoring)
msg = "%f (%f)" % (cv_results.mean(), cv_results.std())
print(msg)

## Play with the hyper-parameters

Different hyer-parameters can be defined. See the list [here](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html). Hereafeter, we will change some of them, and we will test which model has the highest performance score.

In [ ]:
# Initialize models
models = []
models.append(('RF0', RandomForestRegressor()))
models.append(('RF1', RandomForestRegressor(max_features = 3,  n_estimators = 1000)))
models.append(('RF2', RandomForestRegressor(max_features = 3,  n_estimators = 100)))
models.append(('RF3', RandomForestRegressor(max_features = 6,  n_estimators = 1000)))
models.append(('RF4', RandomForestRegressor(max_features = 1,  n_estimators = 100)))
models.append(('RF5', RandomForestRegressor(max_features = 1,  n_estimators = 1000)))

In [ ]:
# CV
results = []
names = []
for name, model in models:
    cv_results = model_selection.cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
    results.append(cv_results)
    names.append(name)
    msg = "%s: %f (%f)" % (name, cv_results.mean(), cv_results.std())
    print(msg)

**QUESTION 3** Plot the histogram of the score values. Which model appears to perform the best? What can you say about the other models? Which hyper-parameter has the largest influence?

In [ ]:
# boxplot algorithm comparison
fig = plt.figure()
fig.suptitle('Algorithm Comparison')
ax = fig.add_subplot(111)
plt.boxplot(results)
ax.set_xticklabels(names)
plt.ylim(0, 1)
plt.show()

listofmeans = [np.mean(i) for i in results]
id = np.argmax(listofmeans)

## More systematic tuning

We use the function [gridsearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) to tune automaticcally the hyper-parameters. 

**QUESTION 4** Choose a list of possible values (replace ## TBC ## by values) of the hyper-parameters and run the code to search for the most optimal configuration.

In [ ]:
# Number of trees in random forest
n_estimators = [## TBC ##]
# Number of features to consider at every split
max_features = [## TBC ##]
# Maximum number of levels in tree
max_depth = [None]
# Minimum number of samples required to split a node
min_samples_split = [## TBC ##]
# Minimum number of samples required at each leaf node
min_samples_leaf = [## TBC ##]
# Method of selecting samples for training each tree
bootstrap = [True]

# Create the random grid
para_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap}
print(para_grid)

#rf_random = RandomizedSearchCV(
rf_random = GridSearchCV(
    estimator = rf0, 
    cv = kfold, 
    param_grid = para_grid,
    verbose = 10,
    scoring =  scoring,
    refit = True
    )

rf_random.fit(X,Y)

**CHALLENGE** Which configuration achieves the best score?

In [ ]:
print(rf_random.best_params_)
print(rf_random.best_score_)
best_model = rf_random.best_estimator_
best_model

## Test alternative models

Now we will test other machine learning models, i.e., [linear regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [support vector machine](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html) for regression, and [gradient boosting](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html).

In [ ]:
models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Support Vector Regressor (SVR)": make_pipeline(StandardScaler(), SVR(kernel='rbf')),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

In [ ]:
# CV
results = []
names = []
for name, model in models.items():
    cv_results = model_selection.cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
    results.append(cv_results)
    names.append(name)
    msg = "%s: %f (%f)" % (name, cv_results.mean(), cv_results.std())
    print(msg)

**QUESTION 5** Which model has the best score? Could you comment also on the value of the standard deviation?

In [ ]:
# boxplot algorithm comparison
fig = plt.figure()
fig.suptitle('Algorithm Comparison')
ax = fig.add_subplot(111)
plt.boxplot(results)
plt.xticks(rotation='vertical')
ax.set_xticklabels(names)
plt.ylim(0, 1)
plt.show()

listofmeans = [np.mean(i) for i in results]
id = np.argmax(listofmeans)

## Tune the most optimal model

**QUESTION 6** Select the best performing model and tune its hyper-parameter. Do not hesitate to extent the range of the proposed list.
*Hints* to tune hyperparameters of a pipeline please adapte the above code by following this [example](https://stackoverflow.com/questions/43366561/use-sklearns-gridsearchcv-with-a-pipeline-preprocessing-just-once).

In [ ]:
sv0 = make_pipeline(StandardScaler(), SVR(kernel='rbf')).fit(X,Y)
C_list = [0.01,,0.1,,1,10,100]## TBC ##
gamma_list = [0.01,0.02,0.1,0.2,0.3,1,2,10]## TBC ##

# Create the random grid
para_grid = {  'svr__gamma': gamma_list,
               'svr__C': C_list}
print(para_grid)

#rf_random = RandomizedSearchCV(
svm_random = GridSearchCV(
    estimator = sv0, 
    cv = kfold, 
    param_grid = para_grid,
    verbose = 1,
    scoring =  scoring,
    refit = True
    )

svm_random.fit(X,Y)

In [ ]:
print(svm_random.best_params_)
print(svm_random.best_score_)
best_model = svm_random.best_estimator_
best_model

**QUESTION 7** Test the same model but without the pre-scaling. To do so, set *svr0* to *SVR(kernel='rbf')*.

## Advanced mode

**QUESTION 8** Perform two new analyses:
* 1. change the number of folds from 5 to 10 and the number of repeatitions from 5 to 1
* 2. change the scoring to 'neg_mean_absolute_error'

For each analysis, note the differences in the conclusions.